In [215]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np
from itertools import combinations

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Galois Field

In [216]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n


Field Closed Succesfully!, 1 Non-Zero Elements


## 1. Codewords Test

### 1.1 Generate all codewords

In [217]:
encoder_output = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))

n_codewords = len(encoder_output)

### 1.2 Decoding of codewords

In [218]:
n_codewords
#encoder_output 
decoder_model = GolayDecoder()

decoder_output = []

for i in range(n_codewords):
    decoder_output.append(decoder_model.correct(encoder_output[i]))

#print(decoder_output)

Field Closed Succesfully!, 1 Non-Zero Elements


### 1.3 Error calculation `o_err` (null, all zeros)

In [219]:
o_err           = []
corrected       = []
uncorrectable   = []

o_corrected     = []
o_uncorrectable = []
o_msg           = []

for i in range(n_codewords):
    decoded_codeword = decoder_output[i][0]
    o_err.append(decoded_codeword - encoder_output[i])

    o_msg.append(decoder_output[i][0][:12])
    o_corrected.append(int(decoder_output[i][1]))
    o_uncorrectable.append(int(decoder_output[i][2]))
    corrected.append(decoder_output[i][1])
    uncorrectable.append(decoder_output[i][2])

print(np.array(o_err).shape)
print(np.array(o_corrected).shape)
print(np.array(o_uncorrectable).shape)
print(np.array(o_msg).shape)

(4096, 24)
(4096,)
(4096,)
(4096, 12)


## 2. Codewords with errors test

### 2.1 Generate errors

In [220]:

encoder_output 

received_msg_with_error = []

n_errors = 5 # 4 errors limit

for n in range(0, n_errors):
    codewords_n_errors = []
    # Select random n positions
    for error_positions in combinations(range(24), n):
        codeword = encoder_output[i].copy()

        # Put errors
        for pos in error_positions:
            codeword[pos] ^= 1

        # Guardar la codeword completa
        codewords_n_errors.append(codeword)

    received_msg_with_error.append(codewords_n_errors)

# Length of all generated error weight 
for i in range(0, 5):
    print(len(received_msg_with_error[i]))

#received_msg_with_error[error_weight][vector]
print(received_msg_with_error[0][0]) #1
# print(received_msg_with_error[1]) #24
# print(received_msg_with_error[2]) #276
# print(received_msg_with_error[3]) #10626

1
24
276
2024
10626
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [ ]:
decoder_array   = []

for i in range(0, len(received_msg_with_error)): # error 0, 1, 2, 3, 4
    decoder_array_n_errors = []
    for rx in received_msg_with_error[i]: # pattern 1, 24, 276, 2024, 10625
        decoder_array_n_errors.append(decoder_model.decode(rx))
    decoder_array.append(decoder_array_n_errors)

o_err_n = []

for i in range(0, len(received_msg_with_error)):
    err_n = []
    for j in range(len(received_msg_with_error[i])):
        rx      = received_msg_with_error[i][j]
        decoded = decoder_array[i][j][0]

        s, q = decoder_model.get_s_q(rx)
        #print(decoder_model.get_error(s, q))
        calculated_error = decoder_model.get_error(s, q)
        if (calculated_error is None):
            err_n.append([0]*24)
        else:
            err_n.append(calculated_error)
    o_err_n.append(err_n)

# print all "calculated error"
print(o_err_n[4])


10626


In [ ]:


# rx
# received_msg_with_error

# msg
# r, _, _ = decoder_model.decode(rx)

# error
# calculate

# corrected
# _, corrected, _ = decoder_model.decode(rx)

# uncorrected
# _, _, uncorrected = decoder_moder.decode(rx)

### 2.1 Decoding of codewords with errors

In [223]:
o_no_cw_err = []
o_no_cw_msg = []
o_no_cw_uncorrectable = []
o_no_cw_corrected = []

for n in range(n_errors - 1):

    # Resultados para esta cantidad de errores
    err_n = []
    msg_n = []
    uncorrectable_n = []
    corrected_n = []

    for i in range(n_codewords):
        received = received_msg_with_error[n][i]
        # Decoder
        decoded = decoder_model.correct(received)
        # Syndrome
        s, q = decoder_model.get_s_q(received)
        # Error pattern
        if decoder_model._gf.do_pack(s) != 0:
            err_n.append(decoder_model.get_error(s, q))
        else:
            err_n.append(0)
        # Message/corrected codeword
        msg_n.append(decoded[0])

        # Flags
        uncorrectable_n.append(int(decoded[1]))
        corrected_n.append(int(decoded[2]))

    # Guardar resultados de este número de errores
    o_no_cw_err.append(err_n)
    o_no_cw_msg.append(msg_n)
    o_no_cw_uncorrectable.append(uncorrectable_n)
    o_no_cw_corrected.append(corrected_n)

IndexError: list index out of range

### Create files `received words with n-errors`

## golay (24,12) decoding example

In [ ]:
r = encoder_output[3576]
# Generate random error for r (word received/transmitted)
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)

# r: word with errors
# word decoded (possible codeword),
# flags (corrected, uncorrectable)
# encoder output (word received transmitted)
r, decoder_model.decode(r), encoder_output[3576]

Field Closed Succesfully!, 1 Non-Zero Elements


(array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1]),
 (array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0]), True, False),
 array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1], dtype=uint8))

In [ ]:
# decode all codewords, no errors
for cw in encoder_output:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_output:
    error_seed      = np.random.randint(0, 0b11111)
    # decimal error_seed converted into n-bits
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    # calculate hamming weight
    error_weight    = decoder_model._gf.hamming_weight(error)
    
    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)
    
    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


## Decoding test with selected values 

Verify the following values:

$$ r_{1} = 0xA5D9A6 $$
$$ r_{2} = 0xA5F9A4 $$
$$ r_{3} = 0xA5C9AA $$

### 1. Obtain values corrected.

In [ ]:
import numpy as np

rx_test = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rx_test_array = [
    np.array([int(bit) for bit in format(x, '024b')], dtype=np.uint8)
    for x in rx_test]

rx_test_array

decoded_rx_test        = []
decoded_rx_test_binary = []

for i in range(len(rx_test_array)):
    decoded_rx_test.append(decoder_model.decode(rx_test_array[i], False))

# print(decoded_rx_test)

for decoded, valid, uncorrectable in decoded_rx_test:
    decoded_rx_test_binary.append(
        (decoded, int(valid), int(uncorrectable))
    )

# decoded word, o_corrected, o_uncorrected
decoded_rx_test_binary

[(array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 0, 1)]

### 2. Obtain mask error

In [ ]:
s_q_vectors  = []
error_vector = []

for i in range (len(rx_test_array)):
    s_q_vectors.append(decoder_model.get_s_q(rx_test_array[i]))
    error_vector.append(decoder_model.get_error(s_q_vectors[i][0], s_q_vectors[i][1]))
    if error_vector[i] is None:
        error_vector[i] = [0]*24
    
error_vector

[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1], dtype=uint8),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1], dtype=uint8),
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [ ]:
from io import StringIO

filenames = [
    "outputs/decoding_vectors_test/codewords_testing_golay_test.svh",
    "../implem/golay_decoder/decoder/decoder.uvm/sequences/decoding_vectors_test/codewords_testing_golay_test.svh"
]
# /home/abel/Desktop/desktop/github_repos/fec-lab2/implem/golay_decoder/decoder/decoder.uvm/sequences
file_content = StringIO()

# Received test vectors
write_sv_array(file_content, "rx_test_vectors", "NB_CODEWORD-1:0", 24,
               [int("".join(map(str, rx)), 2) for rx in rx_test_array],
               comment="Selected received test vectors: 0xA5D9A6, 0xA5F9A4, 0xA5C9AA")

# Decoded message
write_sv_array(file_content, "msg_test_vectors", "NB_WORD-1:0", 12,
               [int("".join(map(str, d[0])), 2) for d in decoded_rx_test_binary],
               comment="Decoded message for the selected test vectors")

# Error vectors
write_sv_array(file_content, "err_pattern_test_vectors", "NB_CODEWORD-1:0", 24,
               [int("".join(map(str, err)), 2) for err in error_vector],
               comment="Recovered error pattern for the selected test vectors")

# Corrected flag
write_sv_array(file_content, "corrected_flag_test_vectors", None, 1,
               [d[1] for d in decoded_rx_test_binary],
               comment="Corrected flag for the selected test vectors")

# Uncorrectable flag
write_sv_array(file_content, "uncorrectable_flag_test_vectors", None, 1,
               [d[2] for d in decoded_rx_test_binary],
               comment="Uncorrectable flag for the selected test vectors")

# Write the same content to both files
content = file_content.getvalue()

for filename in filenames:
    with open(filename, "w") as file:
        file.write(content)